Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

# Solution - Supervisor Agent Pattern in LangGraph

A supervisor agent is a coordinator that repeatedly decides which worker to run next, regains control after each one, and decides when the task is done. It's the router and loop patterns combined: it branches to a worker (router), and workers return to it so it can branch again (loop).

```
START -> [supervisor] --(FINISH)--> END
            |   ^
            v   | (workers loop back)
        [researcher] [writer]
```

The supervisor picks `researcher` first, gets the facts back, then picks `writer`, gets the draft back, then decides the task is complete and routes to `FINISH`. A termination guard guarantees it can't loop forever.

## 0. Install & imports

In [ ]:
# (setup cell already installs what this notebook needs)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

# LLM on the local Ollama bridge
llm = make_llm()

## 1. Define the shared state

The supervisor tracks worker outputs (`research`, `draft`) and its own decision (`next`).

In [ ]:
class SupervisorState(TypedDict, total=False):
    task: str        # input:   what to accomplish
    research: str    # produced by the researcher worker
    draft: str       # produced by the writer worker
    next: str        # supervisor's decision: "researcher" | "writer" | "FINISH"

## 2. Define the supervisor and workers

The supervisor decides the next step (with a termination guard so it always ends); each worker does one job and writes to state.

In [ ]:
def supervisor(state: SupervisorState) -> SupervisorState:
    """Coordinator: pick the next worker, or FINISH when the task is done."""
    has_research = bool(state.get("research"))
    has_draft = bool(state.get("draft"))

    # Termination guard: once both workers have run, always finish.
    # Without this, the LLM could keep re-picking workers forever.
    if has_research and has_draft:
        state["next"] = "FINISH"
        return state

    prompt = (
        "You supervise two workers to complete a task.\n"
        f"Task: {state['task']}\n"
        f"researcher has run: {has_research}. writer has run: {has_draft}.\n"
        "The writer needs the researcher's facts first.\n"
        "Reply with ONE word - the next worker to run: 'researcher' or 'writer'."
    )
    choice = llm.invoke(prompt).content.strip().lower()

    # Normalise, with a sensible fallback (research before writing)
    if "writ" in choice and has_research:
        state["next"] = "writer"
    else:
        state["next"] = "researcher" if not has_research else "writer"
    return state


def researcher(state: SupervisorState) -> SupervisorState:
    prompt = f"List 3 key facts about: {state['task']}"
    state["research"] = llm.invoke(prompt).content
    return state


def writer(state: SupervisorState) -> SupervisorState:
    prompt = f"Using these facts, write a short summary:\n{state.get('research', '')}"
    state["draft"] = llm.invoke(prompt).content
    return state

## 3. Build the graph

Two things define the supervisor pattern here: the conditional edge maps `FINISH` to `END` (router-style dispatch), and the workers loop back to the supervisor instead of ending (the loop).

In [ ]:
builder = StateGraph(SupervisorState)
builder.add_node("supervisor", supervisor)
builder.add_node("researcher", researcher)
builder.add_node("writer", writer)

builder.add_edge(START, "supervisor")


# The supervisor's decision becomes the next node (or END via "FINISH")
def route(state: SupervisorState) -> str:
    return state["next"]


builder.add_conditional_edges(
    "supervisor",
    route,
    {"researcher": "researcher", "writer": "writer", "FINISH": END},
)

# KEY: workers loop BACK to the supervisor (router + loop combined)
builder.add_edge("researcher", "supervisor")
builder.add_edge("writer", "supervisor")

graph = builder.compile()

## 4. Run the supervisor

The team researches, then writes, then the supervisor finishes.

In [ ]:
result = graph.invoke({"task": "the history of the Dutch language"})

print("=== RESEARCH ===\n", result["research"], "\n")
print("=== DRAFT ===\n", result["draft"])

## 5. Bonus - watch the hand-offs

Streaming shows the supervisor regaining control between each worker - the signature rhythm of the pattern: `supervisor -> worker -> supervisor -> worker -> supervisor`.

In [ ]:
# Watch the supervisor hand off and regain control between workers
path = [list(step.keys())[0] for step in graph.stream({"task": "why bees matter"})]
print("nodes run in order:")
print(" -> ".join(path))
# Expect: supervisor -> researcher -> supervisor -> writer -> supervisor (then END)

### Extension ideas

- Add a 3rd worker (e.g. `reviewer` that critiques the draft) - a node, a branch in the guard/decision, and a mapping entry + loop-back edge.
- Let the supervisor use structured output to choose the next worker instead of parsing a word.
- Add a `revisions` counter and let the writer/reviewer loop a bounded number of times - combining the supervisor with the loop-guard pattern explicitly.

### Where this sits

| Pattern | Shape | Edges |
|---|---|---|
| Sequential | fixed line | fixed |
| Loop | repeat a node | conditional (branch back) |
| Router | one-of-N once | conditional (branch out) |
| Supervisor | coordinate N, repeatedly | conditional out + loop back |

This is essentially what `create_agent` does internally: a coordinator (the model) that repeatedly picks a tool, gets the result back, and decides when to finish.